# exp010_trajectory_drift_ablation train

Notebook-first CV ablation for trajectory drift feature variants.


## Contents

1. Setup and configuration
2. Metric, data, and variant helpers
3. CV scoring helpers
4. Fold-safe ablation CV run
5. Metrics and artifacts


## 1. Setup and configuration


In [ ]:
from __future__ import annotations

import json
import os
from datetime import UTC, datetime
from math import sqrt
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold

from baseline import (
    HORIZONTAL_SUFFIX,
    active_feature_columns,
    build_drift_feature_frame,
    config_get,
    drift_strategy,
    fit_drift_model_from_files,
    optional_positive_int,
    predict_drift,
    primary_strategy,
    read_typewell_for_horizontal_path,
    well_id_from_path,
)
from settings import EXPERIMENT_NAME, ExperimentPaths, deep_merge, load_config

DEBUG = os.environ.get("EXPERIMENT_DEBUG", "0") == "1"
MAX_WELLS_ENV = os.environ.get("EXPERIMENT_MAX_WELLS")
MAX_WELLS = int(MAX_WELLS_ENV) if MAX_WELLS_ENV else None
VARIANT_LIMIT_ENV = os.environ.get("EXPERIMENT_VARIANT_LIMIT")
VARIANT_LIMIT = int(VARIANT_LIMIT_ENV) if VARIANT_LIMIT_ENV else None

paths = ExperimentPaths()
paths.require_kaggle_runtime()
paths.ensure_output_dirs()
config = load_config()
primary = primary_strategy(config)
drift_name = drift_strategy(config)

print("Experiment:", EXPERIMENT_NAME)
print("Root:", paths.root)
print("Train data:", paths.train_data_dir)
print("Output root:", paths.output_root)
print("Artifacts:", paths.artifacts_dir)
print("Primary strategy:", primary)
print("Drift strategy:", drift_name)
print("Base feature columns:", len(active_feature_columns(config)))
print("Debug:", DEBUG, "Max wells:", MAX_WELLS, "Variant limit:", VARIANT_LIMIT)


## 2. Metric, Data, And Variant Helpers


In [ ]:
def finite_float(value: float | None, digits: int = 6) -> float | None:
    if value is None or not np.isfinite(value):
        return None
    return round(float(value), digits)


def rmse_from_sums(sse: float, count: int) -> float | None:
    if count <= 0:
        return None
    return sqrt(sse / count)


def rmse_from_arrays(y_true: np.ndarray, y_pred: np.ndarray) -> float | None:
    valid = np.isfinite(y_true) & np.isfinite(y_pred)
    if not valid.any():
        return None
    residual = y_pred[valid] - y_true[valid]
    return float(np.sqrt(np.mean(residual * residual)))


def train_files(paths: ExperimentPaths, cfg: dict[str, Any], debug: bool, max_wells: int | None) -> list[Path]:
    files = sorted(paths.train_data_dir.glob(f"*{HORIZONTAL_SUFFIX}"))
    if not files:
        raise FileNotFoundError(f"no train horizontal well CSVs found in {paths.train_data_dir}")
    if debug:
        limit = max_wells
        if limit is None:
            limit = int(cfg.get("runtime", {}).get("debug_n_wells", 30))
        files = files[:limit]
    elif max_wells is not None:
        files = files[:max_wells]
    return files


def build_fold_map(files: list[Path], n_folds: int) -> tuple[dict[str, int], int]:
    well_ids = [well_id_from_path(path) for path in files]
    n_splits = min(max(1, n_folds), len(well_ids))
    if n_splits < 2:
        raise ValueError("drift residual CV requires at least two wells")

    fold_map: dict[str, int] = {}
    x = np.arange(len(well_ids)).reshape(-1, 1)
    groups = np.asarray(well_ids)
    splitter = GroupKFold(n_splits=n_splits)
    for fold, (_, valid_idx) in enumerate(splitter.split(x, groups=groups)):
        for index in valid_idx:
            fold_map[well_ids[int(index)]] = fold
    return fold_map, n_splits


def configured_row_cap(cfg: dict[str, Any], key: str) -> int | None:
    return optional_positive_int(config_get(cfg, key, None))


def ablation_variants(cfg: dict[str, Any]) -> list[dict[str, Any]]:
    raw_variants = config_get(cfg, "ablation.variants", [])
    if not isinstance(raw_variants, list) or not raw_variants:
        raw_variants = [{"name": "control", "variable": "control", "description": "base config", "overrides": {}}]

    variants: list[dict[str, Any]] = []
    for index, variant in enumerate(raw_variants):
        if not isinstance(variant, dict):
            raise ValueError(f"ablation variant #{index} must be a mapping")
        if variant.get("enabled", True) is False:
            continue
        name = str(variant.get("name") or f"variant_{index}")
        overrides = variant.get("overrides") or {}
        if not isinstance(overrides, dict):
            raise ValueError(f"ablation variant {name} overrides must be a mapping")
        variants.append(
            {
                "name": name,
                "variable": str(variant.get("variable") or "unknown"),
                "description": str(variant.get("description") or ""),
                "overrides": overrides,
            }
        )

    if VARIANT_LIMIT is not None:
        variants = variants[:VARIANT_LIMIT]
    if not variants:
        raise ValueError("no enabled ablation variants")
    return variants


def config_for_variant(base_cfg: dict[str, Any], variant: dict[str, Any]) -> dict[str, Any]:
    variant_cfg = deep_merge(base_cfg, variant["overrides"])
    return variant_cfg


def delta_vs_baseline(value: float | None, baseline: float | None) -> float | None:
    if value is None or baseline is None:
        return None
    return finite_float(float(value) - float(baseline))


## 3. CV scoring helpers


In [ ]:
def add_score(
    stats: dict[str, Any],
    strategy: str,
    fold: int,
    y_true: np.ndarray,
    y_pred: np.ndarray,
) -> None:
    valid = np.isfinite(y_true) & np.isfinite(y_pred)
    if not valid.any():
        return
    residual = y_pred[valid] - y_true[valid]
    sse = float(np.sum(residual * residual))
    count = int(valid.sum())
    stats[strategy]["fold_sse"][fold] += sse
    stats[strategy]["fold_n"][fold] += count
    stats[strategy]["total_sse"] += sse
    stats[strategy]["total_n"] += count
    stats[strategy]["well_rmse"].append(float(np.sqrt(sse / count)))


def summarize_strategy(strategy_stats: dict[str, Any]) -> dict[str, Any]:
    fold_rmse = [
        finite_float(rmse_from_sums(sse, int(count)))
        for sse, count in zip(strategy_stats["fold_sse"], strategy_stats["fold_n"], strict=True)
    ]
    valid_fold_rmse = [value for value in fold_rmse if value is not None]
    well_rmse = np.asarray(strategy_stats["well_rmse"], dtype=float)
    return {
        "oof_rmse": finite_float(
            rmse_from_sums(strategy_stats["total_sse"], int(strategy_stats["total_n"]))
        ),
        "mean_fold_rmse": finite_float(
            float(np.mean(valid_fold_rmse)) if valid_fold_rmse else None
        ),
        "fold_rmse": fold_rmse,
        "rows": int(strategy_stats["total_n"]),
        "per_well_rmse_mean": finite_float(float(np.mean(well_rmse)) if well_rmse.size else None),
        "per_well_rmse_median": finite_float(
            float(np.median(well_rmse)) if well_rmse.size else None
        ),
    }


def first_feature_value(frame: Any, column: str) -> float | None:
    if column not in frame.features.columns or frame.features.empty:
        return None
    return finite_float(frame.features[column].iloc[0])


def write_csv_artifacts(
    paths: ExperimentPaths,
    ablation_records: list[dict[str, Any]],
    well_records: list[dict[str, Any]],
    fold_records: list[dict[str, Any]],
    model_records: list[dict[str, Any]],
) -> None:
    if ablation_records:
        pd.DataFrame(ablation_records).to_csv(
            paths.artifacts_dir / "ablation_metrics.csv", index=False
        )
    if well_records:
        pd.DataFrame(well_records).to_csv(paths.artifacts_dir / "well_metrics.csv", index=False)
    if fold_records:
        pd.DataFrame(fold_records).to_csv(paths.artifacts_dir / "fold_metrics.csv", index=False)
    if model_records:
        pd.DataFrame(model_records).to_csv(
            paths.artifacts_dir / "fold_model_training.csv", index=False
        )


## 4. Fold-Safe Ablation CV Run


In [ ]:
files = train_files(paths, config, debug=DEBUG, max_wells=MAX_WELLS)
fold_map, n_splits = build_fold_map(files, int(config["validation"]["n_folds"]))
variants = ablation_variants(config)
baseline_cv = config_get(config, "ablation.baseline_cv", None)
target_column = config["data"]["target_column"]
seed = int(config["validation"]["seed"])

print("Ablation variants:", [variant["name"] for variant in variants])
print("Wells:", len(files), "Folds:", n_splits)

variant_summaries: dict[str, Any] = {}
ablation_records: list[dict[str, Any]] = []
well_records: list[dict[str, Any]] = []
fold_records: list[dict[str, Any]] = []
model_records: list[dict[str, Any]] = []

for variant_index, variant in enumerate(variants):
    variant_name = variant["name"]
    variant_config = config_for_variant(config, variant)
    variant_primary = primary_strategy(variant_config)
    variant_drift = drift_strategy(variant_config)
    strategy_list = variant_config["model"]["strategies"]
    feature_columns = active_feature_columns(variant_config)
    max_rows_per_fold = configured_row_cap(variant_config, "model.training.max_train_rows_per_fold")
    max_rows_per_well = configured_row_cap(variant_config, "model.training.max_train_rows_per_well")
    max_rows_final = configured_row_cap(variant_config, "model.training.max_train_rows_final")

    print(
        f"Variant {variant_index + 1}/{len(variants)} {variant_name}: "
        f"feature_set={config_get(variant_config, 'model.feature_set', 'all')}, "
        f"features={len(feature_columns)}, max_fold={max_rows_per_fold}, "
        f"max_final={max_rows_final}, max_per_well={max_rows_per_well}, "
        f"shrink={config_get(variant_config, 'model.params.residual_shrink', None)}, "
        f"trajectory_window={config_get(variant_config, 'model.params.trajectory_window', None)}"
    )

    stats = {
        strategy: {
            "fold_sse": np.zeros(n_splits, dtype=float),
            "fold_n": np.zeros(n_splits, dtype=np.int64),
            "total_sse": 0.0,
            "total_n": 0,
            "well_rmse": [],
        }
        for strategy in strategy_list
    }

    for fold in range(n_splits):
        train_fold_files = [path for path in files if fold_map[well_id_from_path(path)] != fold]
        valid_fold_files = [path for path in files if fold_map[well_id_from_path(path)] == fold]
        print(
            f"  Fold {fold}: fitting on {len(train_fold_files)} wells, "
            f"validating on {len(valid_fold_files)} wells"
        )
        model, n_train_rows = fit_drift_model_from_files(
            train_fold_files,
            variant_config,
            seed=seed + fold,
            max_rows_total=max_rows_per_fold,
            max_rows_per_well=max_rows_per_well,
        )
        model_records.append(
            {
                "variant": variant_name,
                "variable": variant["variable"],
                "fold": fold,
                "n_train_wells": len(train_fold_files),
                "n_valid_wells": len(valid_fold_files),
                "n_train_rows": n_train_rows,
                "max_rows_per_fold": max_rows_per_fold,
                "max_rows_final": max_rows_final,
                "max_rows_per_well": max_rows_per_well,
                "feature_set": config_get(variant_config, "model.feature_set", "all"),
                "n_features": len(feature_columns),
                "residual_shrink": config_get(variant_config, "model.params.residual_shrink", None),
                "trajectory_window": config_get(variant_config, "model.params.trajectory_window", None),
            }
        )

        for path in valid_fold_files:
            well_id = well_id_from_path(path)
            df = pd.read_csv(path)
            typewell_df = read_typewell_for_horizontal_path(path)
            frame = build_drift_feature_frame(
                df,
                variant_config,
                include_target=True,
                typewell_df=typewell_df,
                formation_guide=getattr(model, "formation_guide_", None),
            )
            y_true = df.loc[frame.eval_indices, target_column].to_numpy(dtype=float)
            predictions = {
                "last_anchor": frame.baseline_prediction,
                variant_drift: predict_drift(frame, model, variant_config),
            }
            record: dict[str, Any] = {
                "variant": variant_name,
                "variable": variant["variable"],
                "well_id": well_id,
                "fold": fold,
                "n_rows": int(len(df)),
                "n_known": int(frame.last_known_index + 1),
                "n_eval": int(frame.eval_indices.size),
                "last_known_index": frame.last_known_index,
                "last_known_md": frame.last_known_md,
                "last_known_tvt": frame.last_known_tvt,
                "recent_slope": frame.recent_slope,
                "anchor_azimuth_sin": first_feature_value(frame, "anchor_azimuth_sin"),
                "anchor_azimuth_cos": first_feature_value(frame, "anchor_azimuth_cos"),
                "anchor_dz_dmd_minus_prefix_dz_dmd": first_feature_value(frame, "anchor_dz_dmd_minus_prefix_dz_dmd"),
                "anchor_dxy_dmd_minus_prefix_dxy_dmd": first_feature_value(frame, "anchor_dxy_dmd_minus_prefix_dxy_dmd"),
                "target_residual_mean": finite_float(
                    float(np.nanmean(frame.target_residual))
                    if frame.target_residual is not None and frame.target_residual.size
                    else None
                ),
            }
            for strategy in strategy_list:
                y_pred = predictions[strategy]
                add_score(stats, strategy, fold, y_true, y_pred)
                record[f"{strategy}_rmse"] = finite_float(rmse_from_arrays(y_true, y_pred))
            well_records.append(record)

    strategies = {
        strategy: summarize_strategy(strategy_stats)
        for strategy, strategy_stats in stats.items()
    }
    for strategy, strategy_stats in stats.items():
        for fold in range(n_splits):
            fold_records.append(
                {
                    "variant": variant_name,
                    "variable": variant["variable"],
                    "strategy": strategy,
                    "fold": fold,
                    "rmse": finite_float(
                        rmse_from_sums(
                            strategy_stats["fold_sse"][fold],
                            int(strategy_stats["fold_n"][fold]),
                        )
                    ),
                    "rows": int(strategy_stats["fold_n"][fold]),
                }
            )

    primary_metrics = strategies[variant_primary]
    ablation_record = {
        "variant": variant_name,
        "variable": variant["variable"],
        "description": variant["description"],
        "primary_strategy": variant_primary,
        "feature_set": config_get(variant_config, "model.feature_set", "all"),
        "n_features": len(feature_columns),
        "max_rows_per_fold": max_rows_per_fold,
        "max_rows_final": max_rows_final,
        "max_rows_per_well": max_rows_per_well,
        "residual_shrink": config_get(variant_config, "model.params.residual_shrink", None),
        "trajectory_window": config_get(variant_config, "model.params.trajectory_window", None),
        "cv": primary_metrics["oof_rmse"],
        "cv_mean_fold_rmse": primary_metrics["mean_fold_rmse"],
        "rows": primary_metrics["rows"],
        "delta_vs_exp002_cv": delta_vs_baseline(primary_metrics["oof_rmse"], baseline_cv),
    }
    ablation_records.append(ablation_record)
    variant_summaries[variant_name] = {
        **ablation_record,
        "overrides": variant["overrides"],
        "feature_columns": feature_columns,
        "strategies": strategies,
    }
    print(
        f"  {variant_name} {variant_primary} CV RMSE:",
        primary_metrics["oof_rmse"],
        "delta_vs_exp002:",
        ablation_record["delta_vs_exp002_cv"],
    )

write_csv_artifacts(paths, ablation_records, well_records, fold_records, model_records)

ablation_table = pd.DataFrame(ablation_records)
if not ablation_table.empty:
    print(ablation_table.sort_values("cv", na_position="last").to_string(index=False))


## 5. Metrics And Artifacts


In [ ]:
selected_variant = str(config_get(config, "ablation.selected_variant", variants[0]["name"]))
if selected_variant not in variant_summaries:
    print(f"Selected variant {selected_variant} was not run; using {variants[0]['name']} for top-level metrics")
    selected_variant = variants[0]["name"]

selected_summary = variant_summaries[selected_variant]
primary_metrics = selected_summary["strategies"][selected_summary["primary_strategy"]]
completed_records = [record for record in ablation_records if record["cv"] is not None]
best_record = min(completed_records, key=lambda record: record["cv"]) if completed_records else None

metrics = {
    "experiment": EXPERIMENT_NAME,
    "status": "debug_completed" if DEBUG else "completed",
    "created_at": config.get("experiment", {}).get("created_at"),
    "updated_at": datetime.now(UTC).isoformat(),
    "debug": DEBUG,
    "cv": primary_metrics["oof_rmse"],
    "cv_mean_fold_rmse": primary_metrics["mean_fold_rmse"],
    "public_lb": None,
    "private_lb": None,
    "metric": config.get("validation", {}).get("metric"),
    "seed": config.get("validation", {}).get("seed"),
    "primary_strategy": selected_summary["primary_strategy"],
    "selected_variant": selected_variant,
    "best_variant_by_cv": best_record["variant"] if best_record else None,
    "baseline_experiment": config_get(config, "ablation.baseline_experiment", None),
    "baseline_cv": config_get(config, "ablation.baseline_cv", None),
    "n_folds": n_splits,
    "n_wells": len(files),
    "strategies": selected_summary["strategies"],
    "ablation": {
        "records": ablation_records,
        "variants": variant_summaries,
    },
    "model_training": model_records,
    "key_idea": config.get("experiment", {}).get("description"),
    "notes": (
        "Ablation of inference-safe trajectory drift features. "
        "Top-level cv tracks ablation.selected_variant; compare all rows in artifacts/ablation_metrics.csv."
    ),
}
paths.metrics_path.write_text(json.dumps(metrics, indent=2) + "\n")

for record in ablation_records:
    print(f"{record['variant']} CV RMSE:", record["cv"])
print("Selected variant:", selected_variant)
print("Selected CV RMSE:", metrics["cv"])
print("Best variant by CV:", metrics["best_variant_by_cv"])
print("Metrics written:", paths.metrics_path)
